In [ ]:
# Mounting the google drive
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
# Google Drive Path
gDrivePath = "gdrive/MyDrive/Colab Notebooks/COVID19_FaceMaskDetection/"

In [ ]:
import sys
sys.path.append(gDrivePath)

In [ ]:
# import the necessary packages
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras.models import load_model
from utils import Resize
from utils import video_stream
from utils import video_frame
from utils import js_to_image
from google.colab.patches import cv2_imshow
import numpy as np
import argparse
import time
import cv2
import os

In [ ]:
# Setting the Paths
facePath = gDrivePath + "face_detector"
modelPath = gDrivePath + "output/facemask.model"
confidenceVal = 0.5

In [ ]:
# Function to detect and predict whether mask is present on a person's face in a video
def detect_and_predict_mask(frame, faceNet, maskNet):
	# Grabing the dimensions of the frame and then construct a blob from it
	(h, w) = frame.shape[:2]
	blob = cv2.dnn.blobFromImage(frame, 1.0, (300, 300), (104.0, 177.0, 123.0))
 
	# Passing the blob through the network and obtain the face detections
	faceNet.setInput(blob)
	detections = faceNet.forward()
 
	# Initialize the list of faces, their corresponding locations,
	# and the list of predictions from our face mask network
	faces = []
	locs = []
	preds = []
    
    
	# Looping over the detections
	for i in range(0, detections.shape[2]):
		# Extracting the confidence (i.e., probability) associated with the detection
		confidence = detections[0, 0, i, 2]

		# Filtering out weak detections by ensuring the confidence is greater than the minimum confidence
		if confidence > confidenceVal:
			# Computing the (x, y)-coordinates of the bounding box for the object
			box = detections[0, 0, i, 3:7] * np.array([w, h, w, h])
			(startX, startY, endX, endY) = box.astype("int")
	 
			# Ensuring the bounding boxes fall within the dimensions of the frame
			(startX, startY) = (max(0, startX), max(0, startY))
			(endX, endY) = (min(w - 1, endX), min(h - 1, endY))
            
			# Extracting the face ROI, convert it from BGR to RGB channel ordering, resize it to 224x224, and preprocess it
			face = frame[startY:endY, startX:endX]
			face = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
			face = cv2.resize(face, (224, 224))
			face = img_to_array(face)
			face = preprocess_input(face)
	 
			# Adding the face and bounding boxes to their respective lists
			faces.append(face)
			locs.append((startX, startY, endX, endY))
            
	# To only make a predictions if at least one face is detected
	if len(faces) > 0:
		# For faster inference we'll make batch predictions on all faces at the same time rather than one-by-one predictions
		faces = np.array(faces, dtype="float32")
		preds = maskNet.predict(faces, batch_size=32)
	
	# Returning a 2-tuple of the face locations and their corresponding locations
	return (locs, preds)

In [ ]:
# Loading our serialized face detector model from disk
print("Loading face detector model...")
prototxtPath = os.path.sep.join([facePath, "deploy.prototxt"])
weightsPath = os.path.sep.join([facePath, "res10_300x300_ssd_iter_140000.caffemodel"])
faceNet = cv2.dnn.readNet(prototxtPath, weightsPath)

# Loading the face mask detector model from disk
print("Loading our face mask detector model...")
maskNet = load_model(modelPath)

Loading face detector model...
Loading our face mask detector model...


In [ ]:
# Initialize and start the video stream
print("Starting the video stream...")
video_stream()
label_html = 'Capturing...'
bbox = ''

# Looping over the frames from the video stream
while True:
	vs = video_frame(label_html, bbox)
	frame = js_to_image(vs["img"])
 
	# Grabbing the frame from the threaded video stream and resize it to have a maximum width of 400 pixels
	# frame = vs.read()
	frame = Resize(frame, width=400)
	# Detecting faces in the frame and determining whether they are wearing a face mask or not
	(locs, preds) = detect_and_predict_mask(frame, faceNet, maskNet)
    
	# Looping over the detected face locations and their corresponding locations
	for (box, pred) in zip(locs, preds):
		# Unpacking the bounding box and predictions
		(startX, startY, endX, endY) = box
		(mask, withoutMask) = pred

		# Determining the class label and color we'll use to draw the bounding box and text
		label = "Mask" if mask > withoutMask else "No Mask"
		color = (0, 255, 0) if label == "Mask" else (0, 0, 255)
	
		# Including the probability in the label
		label = "{}: {:.2f}%".format(label, max(mask, withoutMask) * 100)
	
		# Displaying the label and bounding box rectangle on the output frame
		cv2.putText(frame, label, (startX, startY - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 2)
		cv2.rectangle(frame, (startX, startY), (endX, endY), color, 2)
    
	# Showing the output frame
	cv2_imshow(frame)
	key = cv2.waitKey(1) & 0xFF
	time.sleep(4.0)
	
	# If the 'q' key was pressed, break from the loop
	if key == ord("q"):
		break
		
# Cleanup code
cv2.destroyAllWindows()
vs.stop()